# 3 — Enhanced Classifier Extraction

Uses Azure AI Content Understanding's **Enhanced Classifier** pattern:
1. Deploy a custom **FS extraction analyzer** (reuses the `sec_financial_tables_v1` schema)
2. Deploy an **enhanced classifier** with `enableSegment: true` and 6 content categories
   (5 FS types + Other), each FS category linked to the extraction analyzer
3. Process each PDF in a single call — CU segments, classifies, and extracts automatically

No manual page detection required.

In [ ]:
import json, os, re, time
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.contentunderstanding import ContentUnderstandingClient

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")

endpoint = os.environ["FOUNDRY_ENDPOINT"].split("/api/projects/")[0].rstrip("/") + "/"
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"],
)
client = ContentUnderstandingClient(endpoint=endpoint, credential=credential, api_version="2025-11-01")
print("Client ready:", endpoint)

## Deploy Analyzer & Classifier

In [ ]:
# Deploy the financial-table extraction analyzer (reuse existing schema)
ANALYZER_ID = "sec_financial_tables_v1"
template_path = REPO_ROOT / "analyzers" / f"{ANALYZER_ID}.json"
tmpl = json.loads(template_path.read_text(encoding="utf-8"))
tmpl["models"] = {
    "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
    "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
}

t0 = time.time()
client.begin_create_analyzer(analyzer_id=ANALYZER_ID, resource=tmpl, allow_replace=True).result()
print(f"Analyzer '{ANALYZER_ID}' deployed in {time.time() - t0:.1f}s")

In [ ]:
# Deploy the enhanced classifier with per-FS-type categories
CLASSIFIER_ID = "sec_fs_classifier_v1"

classifier_template = {
    "baseAnalyzerId": "prebuilt-document",
    "description": (
        "SEC 10-K/10-Q financial statement classifier. "
        "Segments filings into the five primary consolidated financial statements "
        "and routes each to a custom extraction analyzer."
    ),
    "config": {
        "returnDetails": True,
        "enableSegment": True,
        "contentCategories": {
            "BalanceSheet": {
                "description": (
                    "Consolidated Balance Sheets or Statement of Financial Position. "
                    "Shows total assets, total liabilities, and stockholders' equity "
                    "at specific reporting dates. Typically titled "
                    "'Consolidated Balance Sheets'."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "IncomeStatement": {
                "description": (
                    "Consolidated Statements of Operations, Statements of Income, "
                    "or Income Statements. Shows revenues, costs, expenses, and "
                    "net income/loss over reporting periods. "
                    "Does NOT include Statements of Comprehensive Income."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "ComprehensiveIncome": {
                "description": (
                    "Consolidated Statements of Comprehensive Income (Loss). "
                    "Shows net income plus other comprehensive income items such as "
                    "unrealized gains/losses, foreign currency translation, "
                    "and pension adjustments."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "Equity": {
                "description": (
                    "Consolidated Statements of Changes in Stockholders' Equity. "
                    "Shows changes in equity components (common stock, APIC, "
                    "retained earnings, AOCI, treasury stock) across periods. "
                    "May contain multiple sub-tables for different comparison periods."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "CashFlow": {
                "description": (
                    "Consolidated Statements of Cash Flows. "
                    "Shows operating, investing, and financing activities "
                    "with beginning and ending cash balances."
                ),
                "analyzerId": ANALYZER_ID,
            },
            "Other": {
                "description": (
                    "Any page that is NOT one of the five primary financial "
                    "statement tables. Includes: cover pages, table of contents, "
                    "MD&A, notes to financial statements, auditor reports, "
                    "exhibits, and all other content."
                ),
            },
        },
    },
    "models": {
        "completion": os.environ["GPT41_MODEL_DEPLOYMENT"],
        "embedding": os.environ["EMBEDDING_MODEL_DEPLOYMENT"],
    },
    "tags": {"purpose": "sec-fs-classifier", "version": "v1"},
}

t0 = time.time()
client.begin_create_analyzer(
    analyzer_id=CLASSIFIER_ID, resource=classifier_template, allow_replace=True
).result()
print(f"Classifier '{CLASSIFIER_ID}' deployed in {time.time() - t0:.1f}s")

## Classify & Extract

Process all PDFs through the enhanced classifier.
Each PDF is sent in a single call — CU handles segmentation, classification,
and per-segment field extraction.

In [ ]:
pdf_dir = REPO_ROOT / "email" / "attachements"
pdfs = sorted(pdf_dir.glob("*.pdf"))
print(f"Found {len(pdfs)} PDFs\n")

classifier_results = {}
for pdf in pdfs:
    print(f"  {pdf.name} ...", end=" ", flush=True)
    t0 = time.time()
    poller = client.begin_analyze_binary(
        analyzer_id=CLASSIFIER_ID,
        binary_input=pdf.read_bytes(),
        content_type="application/pdf",
    )
    result = poller.result().as_dict()
    classifier_results[pdf.stem] = result
    elapsed = time.time() - t0

    contents = result.get("contents", [])
    cats = {}
    for seg in contents:
        cat = seg.get("category", "Unknown")
        cats[cat] = cats.get(cat, 0) + 1
    cat_str = ", ".join(f"{k}:{v}" for k, v in sorted(cats.items()))
    print(f"{len(contents)} segments, {elapsed:.1f}s  [{cat_str}]")

print("\nDone.")

In [ ]:
# Display classification and extraction results
EXPECTED_FS = ["BalanceSheet", "IncomeStatement", "ComprehensiveIncome", "Equity", "CashFlow"]

for stem, result in classifier_results.items():
    contents = result.get("contents", [])
    print(f"\n{'='*80}")
    print(f"{stem}")
    print(f"{'='*80}")

    for seg in contents:
        cat = seg.get("category", "Unknown")
        if cat == "Other":
            continue
        pg_start = seg.get("startPageNumber", "?")
        pg_end = seg.get("endPageNumber", "?")
        fields = seg.get("fields", {})
        ft = fields.get("financialTables", {})
        tables = ft.get("valueArray", [])
        print(f"\n  {cat} (pages {pg_start}-{pg_end}): {len(tables)} table(s)")
        for j, tbl_raw in enumerate(tables, 1):
            tobj = tbl_raw.get("valueObject", {})
            title = (tobj.get("tableTitle") or {}).get("valueString", "")
            stype = (tobj.get("statementType") or {}).get("valueString", "")
            rows = (tobj.get("rows") or {}).get("valueArray", [])
            hdrs = (tobj.get("periodHeaders") or {}).get("valueArray", [])
            unit = (tobj.get("unit") or {}).get("valueString", "")
            print(f"    T{j}: {stype} | {len(rows)} rows, {len(hdrs)} cols | {unit} | {title[:60]}")

    found = {seg.get("category") for seg in contents if seg.get("category") != "Other"}
    missing = [fs for fs in EXPECTED_FS if fs not in found]
    if missing:
        print(f"\n  MISSING: {', '.join(missing)}")
    else:
        print(f"\n  All 5 FS types found")

## Save & Export

In [ ]:
import sys
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from export_to_excel import export_document

out_dir = REPO_ROOT / "output"
out_dir.mkdir(exist_ok=True)
excel_dir = out_dir / "excel"

for stem, result in classifier_results.items():
    # Save raw classifier result (includes segment metadata)
    raw_path = out_dir / f"{stem}_classified_raw.json"
    raw_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")

    # Merge all FS segments' financialTables into one list for export
    merged_tables = []
    for seg in result.get("contents", []):
        if seg.get("category", "Other") == "Other":
            continue
        ft = seg.get("fields", {}).get("financialTables", {})
        merged_tables.extend(ft.get("valueArray", []))

    merged = {
        "contents": [{
            "fields": {
                "financialTables": {
                    "type": "array",
                    "valueArray": merged_tables,
                }
            }
        }]
    }
    json_path = out_dir / f"{stem}_classified.json"
    json_path.write_text(json.dumps(merged, indent=2, ensure_ascii=False), encoding="utf-8")

    # Export to Excel
    xlsx_path = export_document(json_path, excel_dir)
    print(f"{stem}: {len(merged_tables)} tables -> {json_path.name}, {xlsx_path.name}")